# পুনরাবৃত্ত নিউরাল নেটওয়ার্ক

আগের মডিউলে, আমরা টেক্সটের সমৃদ্ধ অর্থগত উপস্থাপনা নিয়ে আলোচনা করেছি। আমরা যে স্থাপত্যটি ব্যবহার করছি তা বাক্যের শব্দগুলির সম্মিলিত অর্থ ধারণ করে, তবে এটি শব্দগুলির **ক্রম** বিবেচনায় রাখে না, কারণ এম্বেডিংগুলির পরবর্তী সংক্ষিপ্তকরণ অপারেশনটি মূল টেক্সট থেকে এই তথ্যটি সরিয়ে দেয়। যেহেতু এই মডেলগুলি শব্দের ক্রম উপস্থাপন করতে অক্ষম, তারা আরও জটিল বা দ্ব্যর্থক কাজ যেমন টেক্সট জেনারেশন বা প্রশ্নোত্তর সমাধান করতে পারে না।

একটি টেক্সট ক্রমের অর্থ ধারণ করার জন্য, আমরা একটি নিউরাল নেটওয়ার্ক স্থাপত্য ব্যবহার করব যার নাম **পুনরাবৃত্ত নিউরাল নেটওয়ার্ক**, বা RNN। RNN ব্যবহার করার সময়, আমরা আমাদের বাক্যটিকে একসঙ্গে একটিমাত্র টোকেনের মাধ্যমে নেটওয়ার্কে পাঠাই, এবং নেটওয়ার্ক কিছু **স্টেট** উৎপাদন করে যা আমরা পরবর্তী টোকেনের সাথে আবার নেটওয়ার্কে পাঠাই।

![একটি উদাহরণ পুনরাবৃত্ত নিউরাল নেটওয়ার্কের জেনারেশন দেখাচ্ছে।](../../../../../translated_images/bn/rnn.27f5c29c53d727b5.webp)

টোকেনের ইনপুট ক্রম $X_0,\dots,X_n$ দেওয়া হলে, RNN একটি নিউরাল নেটওয়ার্ক ব্লকের ক্রম তৈরি করে, এবং এই ক্রমটি ব্যাকপ্রপ্যাগেশন ব্যবহার করে এন্ড-টু-এন্ড প্রশিক্ষণ দেয়। প্রতিটি নেটওয়ার্ক ব্লক একটি জোড়া $(X_i,S_i)$ ইনপুট হিসেবে নেয়, এবং ফলস্বরূপ $S_{i+1}$ উৎপন্ন করে। সর্বশেষ স্টেট $S_n$ বা আউটপুট $Y_n$ একটি লিনিয়ার ক্লাসিফায়ারে প্রবেশ করানো হয় ফলাফল উত্পাদনের জন্য। সমস্ত নেটওয়ার্ক ব্লক একই ওয়েট শেয়ার করে, এবং একটি ব্যাকপ্রপ পাস ব্যবহার করে এন্ড-টু-এন্ড প্রশিক্ষণ হয়।

> উপরের চিত্রে পুনরাবৃত্ত নিউরাল নেটওয়ার্ককে আনরোল্ড ফর্মে (বাম পাশে) এবং আরও সংক্ষিপ্ত পুনরাবৃত্ত উপস্থাপনায় (ডান পাশে) দেখানো হয়েছে। এটি বোঝা গুরুত্বপূর্ণ যে সমস্ত RNN সেল একই **শেয়ারযোগ্য ওয়েট** ব্যবহার করে।

যেহেতু স্টেট ভেক্টরগুলি $S_0,\dots,S_n$ নেটওয়ার্কের মাধ্যমে পাঠানো হয়, RNN শব্দগুলির মধ্যে ক্রমাগত নির্ভরশীলতা শিখতে সক্ষম। উদাহরণস্বরূপ, যখন ক্রমের মধ্যে কোথাও শব্দ *not* উপস্থিত হয়, এটি স্টেট ভেক্টরের নির্দিষ্ট উপাদানগুলিকে নেগেট করার জন্য শিখতে পারে।

ভিতরে, প্রতিটি RNN সেল দুইটি ওয়েট ম্যাট্রিক্স ধারণ করে: $W_H$ এবং $W_I$, এবং বায়াস $b$। প্রতি RNN ধাপে, ইনপুট $X_i$ এবং ইনপুট স্টেট $S_i$ দেওয়া হলে, আউটপুট স্টেট হিসাব করা হয় $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, যেখানে $f$ একটি অ্যাক্টিভেশন ফাংশন (সাধারণত $\tanh$)।

> টেক্সট জেনারেশন (যা আমরা পরবর্তী ইউনিটে আলোচনা করব) বা মেশিন অনুবাদের মতো সমস্যার জন্য আমরা প্রতিটি RNN ধাপে কিছু আউটপুট মানও পেতে চাই। এই ক্ষেত্রে, আরেকটি ম্যাট্রিক্স $W_O$ থাকে, এবং আউটপুট হিসাব করা হয় $Y_i=f(W_O\times S_i+b_O)$ হিসেবে।

চলুন দেখি কীভাবে পুনরাবৃত্ত নিউরাল নেটওয়ার্ক আমাদের খবরের ডেটাসেট শ্রেণিবদ্ধ করতে সাহায্য করতে পারে।

> স্যান্ডবক্স পরিবেশের জন্য, আমাদের নীচের সেলটি চালাতে হবে যাতে প্রয়োজনীয় লাইব্রেরি ইনস্টল থাকে এবং ডেটা প্রিফেচ করা থাকে। যদি আপনি স্থানীয়ভাবে চালান, তবে নিম্নলিখিত সেলটি উপেক্ষা করতে পারেন।


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

বড় মডেল প্রশিক্ষণের সময়, GPU মেমরি বরাদ্দ একটি সমস্যা হতে পারে। আমাদের বিভিন্ন মিনিব্যাচ আকার নিয়ে পরীক্ষা করতে হতে পারে, যাতে ডেটা আমাদের GPU মেমরিতে ফিট হয়, তবুও প্রশিক্ষণ যথেষ্ট দ্রুত হয়। আপনি যদি এই কোডটি আপনার নিজস্ব GPU মেশিনে চালাচ্ছেন, তবে প্রশিক্ষণের গতি বাড়ানোর জন্য মিনিব্যাচ আকার সামঞ্জস্য করার সঙ্গে পরীক্ষা করতে পারেন।

> **নোট**: NVidia ড্রাইভারের কিছু নির্দিষ্ট সংস্করণ প্রশিক্ষণ শেষে মেমরি মুক্ত না করার জন্য পরিচিত। আমরা এই নোটবুকগুলিতে বেশ কিছু উদাহরণ চালাচ্ছি, এবং এটি নির্দিষ্ট সেটআপে মেমরি শেষ হয়ে যেতে পারে, বিশেষ করে যদি আপনি একই নোটবুকের অংশ হিসেবে নিজস্ব পরীক্ষা চালাচ্ছেন। যদি মডেল প্রশিক্ষণ শুরু করার সময় কিছু অদ্ভুত ত্রুটি দেখা দেয়, তবে আপনি নোটবুক কার্নেল পুনরায় চালু করতে চাইতে পারেন।


In [3]:
batch_size = 16
embed_size = 64

## সহজ RNN শ্রেণীবিন্যাসকারী

একটি সহজ RNN-এর ক্ষেত্রে, প্রতিটি রিকারেন্ট ইউনিট একটি সহজ লিনিয়ার নেটওয়ার্ক, যা একটি ইনপুট ভেক্টর এবং স্টেট ভেক্টর গ্রহণ করে, এবং একটি নতুন স্টেট ভেক্টর উৎপন্ন করে। Keras-এ, এটি `SimpleRNN` লেয়ার দ্বারা উপস্থাপন করা যেতে পারে।

যদিও আমরা RNN লেয়ারে সরাসরি ওয়ান-হট এনকোড করা টোকেনগুলি প্রেরণ করতে পারি, উচ্চ মাত্রিকতার কারণে এটি ভাল ধারণা না। অতএব, আমরা একটি এমবেডিং লেয়ার ব্যবহার করব শব্দ ভেক্টরগুলির মাত্রা হ্রাস করতে, যার পরে একটি RNN লেয়ার থাকবে, এবং শেষমেশ একটি `Dense` শ্রেণীবিন্যাসকারী থাকবে।

> **বিঃদ্রঃ**: যেখানে মাত্রিকতা এত বেশি নয়, যেমন ক্যারেক্টার-লেভেল টোকেনাইজেশন ব্যবহার করার সময়, সেখানে ওয়ান-হট এনকোড করা টোকেনগুলি সরাসরি RNN সেলে প্রেরণ করা যুক্তিযুক্ত হতে পারে।


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **নোট:** আমরা এখানে সরলতার জন্য একটি অনপ্রশিক্ষিত এমবেডিং লেয়ার ব্যবহার করছি, কিন্তু আরও ভাল ফলাফলের জন্য আমরা পূর্ববর্তী ইউনিটে বর্ণিত Word2Vec ব্যবহার করে একটি pretrained এমবেডিং লেয়ার ব্যবহার করতে পারি। এই কোডটিকে pretrained এমবেডিংয়ের সাথে কাজ করার জন্য মানিয়ে নেওয়া আপনার জন্য একটি ভালো অনুশীলন হতে পারে।

এখন আসুন আমাদের RNN প্রশিক্ষণ করি। সাধারণত RNN প্রশিক্ষণ দেওয়া বেশ কঠিন, কারণ একবার RNN সেলগুলি সিকোয়েন্স দৈর্ঘ্য বরাবর আনরোল করা হলে, ব্যাকপ্রোপাগেশনে জড়িত স্তরগুলির সংখ্যা অনেক বেশি হয়। তাই আমাদের একটি ছোট শেখার হার নির্বাচন করতে হয়, এবং ভাল ফলাফল পেতে বড় ডেটাসেটে নেটওয়ার্কটিকে প্রশিক্ষণ দিতে হয়। এতে বেশ সময় লাগতে পারে, তাই GPU ব্যবহার করা উত্তম।

গতি বাড়ানোর জন্য, আমরা কেবল নিউজ শিরোনামে RNN মডেলটি প্রশিক্ষণ দেব, বিবরণ অংশটি বাদ দিয়ে। আপনি চাইলে বিবরণ অংশসহ প্রশিক্ষণ দিয়ে দেখতে পারেন এবং দেখতে পারেন মডেলটি প্রশিক্ষিত হচ্ছে কিনা।


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **নোট** যে এখানে সঠিকতা সম্ভবত কম হবে, কারণ আমরা শুধুমাত্র সংবাদ শিরোনামে প্রশিক্ষণ দিচ্ছি।


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


যেহেতু আমরা মাক্সিং ব্যবহার করছি, তাই আমরা শিরোনাম এবং বিবরণের সম্পূর্ণ ডেটাসেটের উপর মডেল প্রশিক্ষণ দিতে পারি।

> **নোট**: আপনি কি লক্ষ্য করেছেন যে আমরা একটি ভেক্টরাইজার ব্যবহার করছি যা খবরের শিরোনামের উপর প্রশিক্ষিত, পুরো নিবন্ধের শরীরের উপর নয়? সম্ভবত, এর ফলে কিছু টোকেন উপেক্ষিত হতে পারে, তাই ভেক্টরাইজার পুনরায় প্রশিক্ষণ দেওয়া ভালো। তবে এর প্রভাব খুবই সামান্য হতে পারে, তাই সরলতার জন্য আমরা আগের প্রি-ট্রেইনড ভেক্টরাইজারই ব্যবহার করব।


## LSTM: দীর্ঘ স্বল্পমেয়াদি স্মৃতি

RNN-এর প্রধান সমস্যা গুলোর একটি হল **ধূলোমান গ্র্যাডিয়েন্টস**। RNN গুলো বেশ দীর্ঘ হতে পারে, এবং ব্যাকপ্রোপাগেশনের সময় গ্র্যাডিয়েন্টসকে নেটওয়ার্কের প্রথম স্তর পর্যন্ত পৌঁছানো কঠিন হতে পারে। যখন এটি ঘটে, তখন নেটওয়ার্ক দূরের টোকেনগুলোর মধ্যে সম্পর্ক শিখতে পারে না। এই সমস্যা এড়ানোর একটি উপায় হল **স্পষ্ট স্টেট ম্যানেজমেন্ট** পরিচয় করিয়ে দেওয়া **গেটস** ব্যবহার করে। গেটস পরিচয় করিয়ে দেওয়া সবচেয়ে সাধারণ দুটি আর্কিটেকচার হল **দীর্ঘ স্বল্পমেয়াদি স্মৃতি** (LSTM) এবং **গেটেড রিলে ইউনিট** (GRU)। এখানে আমরা LSTM নিয়ে আলোচনা করব।

![একটি উদাহরণ দীর্ঘ স্বল্পমেয়াদি স্মৃতি সেলের ছবি](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

একটি LSTM নেটওয়ার্ক RNN এর মতো সংগঠিত, কিন্তু এখানে দুইটি স্টেট স্তর থেকে স্তরে চলে: আসল স্টেট $c$, এবং লুকানো ভেক্টর $h$। প্রতিটি ইউনিটে, লুকানো ভেক্টর $h_{t-1}$ ইনপুট $x_t$ এর সাথে মিলিত হয়, এবং একসাথে তারা স্টেট $c_t$ এবং আউটপুট $h_{t}$ কী হবে তা **গেটস** এর মাধ্যমে নিয়ন্ত্রণ করে। প্রতিটি গেটের সিগময়েড অ্যানেকটিভেশন থাকে (আউটপুট যা $[0,1]$ সীমার মধ্যে থাকে), যা স্টেট ভেক্টরের সাথে গুণ করার সময় বিটওয়াইজ মাস্ক হিসেবে ভাবা যায়। LSTM এ নিচের গেটগুলো থাকে (ছবির বাম থেকে ডানে):
* **ভুলে যাওয়ার গেট** যা নির্ধারণ করে কোন অংশগুলো ভেক্টর $c_{t-1}$ থেকে ভুলে যেতে হবে, এবং কোন অংশগুলো পার হতে হবে।
* **ইনপুট গেট** যা নির্ধারণ করে ইনপুট ভেক্টর এবং পূর্ববর্তী লুকানো ভেক্টর থেকে কতটুকু তথ্য স্টেট ভেক্টরে অন্তর্ভুক্ত করতে হবে।
* **আউটপুট গেট** যা নতুন স্টেট ভেক্টর নিয়ে সিদ্ধান্ত নেয় এর কোন অংশগুলো দিয়ে নতুন লুকানো ভেক্টর $h_t$ তৈরি হবে।

স্টেট $c$ এর উপাদানগুলোকে ফ্ল্যাগ হিসেবে ভাবা যায় যা চালু এবং বন্ধ করা যায়। উদাহরণস্বরূপ, যখন আমরা সিকোয়েন্সে *Alice* নাম দেখি, আমরা অনুমান করি যে এটি একটি মহিলা নির্দেশ করে, এবং স্টেটের সেই ফ্ল্যাগ চালু করি যা বলে আমাদের বাক্যে একটি নারীসত্বা রয়েছে। যখন আমরা আরও শব্দ *and Tom* পাই, তখন আমরা ফ্ল্যাগ চালু করি যা বলে আমাদের বাক্যে বহুবচন সম্বোধক রয়েছে। তাই স্টেট নিয়ন্ত্রণ করে আমরা বাক্যের ব্যাকরণগত গুণাবলী ট্র্যাক করতে পারি।

> **দ্রষ্টব্য**: LSTM এর অভ্যন্তরীণ ধারণা বোঝার জন্য একটি দুর্দান্ত সূত্র হলো: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) ক্রিস্টোফার ওলাহ কর্তৃক।

যদিও একটি LSTM সেলের অভ্যন্তরীণ গঠন জটিল মনে হতে পারে, Keras এটি `LSTM` লেয়ারের ভিতরে লুকিয়ে রাখে, তাই উপরে উদাহরণে আমাদের যা করার দরকার কেবল রিকারেন্ট লেয়ার প্রতিস্থাপন করা:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **মন্তব্য** যে LSTMs প্রশিক্ষণ করানোও বেশ ধীর, এবং প্রশিক্ষণের শুরুতে আপনাকে অনেক বেশি সঠিকতা বৃদ্ধির খবর নাও দিতে পারে। ভাল সঠিকতা অর্জনের জন্য আপনাকে কিছু সময় ধরে প্রশিক্ষণ চালিয়ে যেতে হতে পারে।


## দ্বিমুখী এবং বহস্তর RNNs

আমাদের উদাহরণগুলিতে এখন পর্যন্ত, পুনরাবৃত্ত নেটওয়ার্কগুলি একটি সিকোয়েন্সের শুরু থেকে শেষ পর্যন্ত কাজ করে। এটা আমাদের কাছে স্বাভাবিক মনে হয় কারণ এটি আমাদের পড়ার বা ভাষণ শোনার একই দিক অনুসরণ করে। তবে, ইনপুট সিকোয়েন্সের র্যান্ডম অ্যাক্সেসের জন্য, পুনরাবৃত্ত গণনা উভয় দিকে চালানো আরও যুক্তিযুক্ত। RNNs যেগুলো উভয় দিক থেকে গণনা করার অনুমতি দেয় সেগুলোকে **দ্বিমুখী** RNN বলা হয়, এবং এগুলো একটি বিশেষ `Bidirectonal` স্তর দিয়ে পুনরাবৃত্ত স্তরটি আবৃত করে তৈরি করা যেতে পারে।

> **বিঃদ্রঃ**: `Bidirectional` স্তরটির মধ্যে স্তরটির দুটি অনুলিপি তৈরি করে এবং সেগুলির একটি অনুলিপির `go_backwards` বৈশিষ্ট্যটি `True` এ সেট করে, যা এটিকে সিকোয়েন্সের বিপরীত দিকে নিয়ে যায়।

পুনরাবৃত্ত নেটওয়ার্ক, একমুখী বা দ্বিমুখী, একটি সিকোয়েন্সের মধ্যে প্যাটার্ন ক্যাপচার করে এবং সেগুলোকে স্টেট ভেক্টরে সংরক্ষণ করে বা আউটপুট হিসেবে ফিরিয়ে দেয়। কনভলিউশানাল নেটওয়ার্কের মতোই, আমরা প্রথমটির পরে অন্য একটি পুনরাবৃত্ত স্তর তৈরি করতে পারি উচ্চ স্তরের প্যাটার্ন ক্যাপচার করার জন্য, যা প্রথম স্তর দ্বারা উত্তোলিত নিম্ন স্তরের প্যাটার্ন থেকে গঠিত। এটি আমাদের **বহস্তর RNN** ধারণার দিকে নিয়ে যায়, যা দুই বা ততোধিক পুনরাবৃত্ত নেটওয়ার্ক নিয়ে গঠিত, যেখানে পূর্ববর্তী স্তরের আউটপুট পরবর্তী স্তরের ইনপুট হিসেবে প্রেরণ করা হয়।

![একটি বহস্তর দীর্ঘমেয়াদী-সংক্ষিপ্তকালীন-মেমোরি RNN দেখানো ছবি](../../../../../translated_images/bn/multi-layer-lstm.dd975e29bb2a59fe.webp)

*ছবিটি [এই চমৎকার পোস্ট](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) থেকে ফার্নান্দো লোপেজ দ্বারা।*

Keras এই নেটওয়ার্কগুলি নির্মাণ করা সহজ করে দেয়, কারণ আপনাকে মডেলটিতে শুধু আরো পুনরাবৃত্ত স্তর যোগ করতে হবে। শেষ স্তর ছাড়া সব স্তরের জন্য, আমাদের `return_sequences=True` প্যারামিটার নির্দিষ্ট করতে হবে, কারণ আমাদের স্তরটি সব মধ্যবর্তী স্টেট ফেরত দিতে হবে, শুধুমাত্র পুনরাবৃত্ত গণনার চূড়ান্ত স্টেট নয়।

চলুন আমাদের শ্রেণীবিন্যাস সমস্যার জন্য একটি দ্বিমুখী দুস্তর LSTM তৈরি করি।

> **বিঃদ্রঃ** এই কোডটি আবার ভীষণ সময় নেয় সম্পন্ন হতে, তবে এটি এপর্যন্ত দেখা সর্বোচ্চ সঠিকতা দেয়। সুতরাং, হয়তো এটি অপেক্ষা করার এবং ফলাফল দেখার যোগ্য।


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## অন্যান্য কাজের জন্য RNNs

এখন পর্যন্ত, আমরা RNN ব্যবহার করে টেক্সটের সিকোয়েন্স শ্রেণীবদ্ধ করার দিকে মনোনিবেশ করেছি। কিন্তু তারা আরো অনেক কাজ পরিচালনা করতে পারে, যেমন টেক্সট জেনারেশন এবং মেশিন অনুবাদ — আমরা পরবর্তী ইউনিটে এই কাজগুলো বিবেচনা করব।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
